# [3장 2강] - 로지스틱 회귀와 교차 검증
## 실습 목표
--- 
- 이진 분류로 꽃받침 데이터를 활용해 두 가지 붓꽃 품종을 구분할 수 있다.
- 다중 분류 및 K-폴드 검증으로 전체 품종을 분류하고 모델 안정성을 평가할 수 있다.
- 특성 변수 재학습을 통해 변수 개수가 모델 성능에 미치는 영향을 비교할 수 있다.

### 필수 1 : 꽃받침 크기를 활용한 붓꽃의 이진 분류
#### 문제 1-1 :  꽃받침 데이터 필터링 및 이진 로지스틱 회귀 모델 학습

**요구 사항**

---

1. 판다스(Pandas) 라이브러리를 사용해 제공된 붓꽃 데이터셋(Iris.csv)을 불러오세요.
2. 품종(Species) 열에서 버시컬러(Iris-versicolor)와 버지니카(Iris-virginica)에 해당하는 데이터만 필터링하여 복사본 데이터 세트를 생성하세요.
3. 입력 변수로는 꽃받침 길이(SepalLengthCm)와 꽃받침 너비(SepalWidthCm) 열을 선택하고, 정답 타깃 변수는 품종 이름을 각각 0과 1의 숫자로 변환하여 설정하세요.
4. 데이터를 훈련 세트 80%, 테스트 세트 20% 비율로 분할하세요. 이때 난수 고정값(random_state)은 42로 지정하고 층화 추출(stratify)을 적용하세요.
5. 로지스틱 회귀 모델 객체를 생성하고 훈련 세트를 사용하여 모델 학습을 진행하세요.
6. 학습이 완료되면 테스트 세트를 활용하여 최종 예측을 수행하고 모델의 최종 테스트 정확도를 계산하여 출력하세요.

**출력 및 검증 방법**
출력 창에 계산된 이진 분류의 최종 테스트 정확도가 소수점 넷째 자리까지 표시되는지 확인하세요. 정확도 결과가 출력된 영역을 캡처하여 작성된 코드와 함께 제출하세요.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score

df = pd.read_csv('Iris.csv')
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
x = df[df['Species'].isin(['Iris-versicolor','Iris-virginica'])].copy()
xx = x[['SepalLengthCm' ,'SepalWidthCm']]

xxx = x['Species'].map({'Iris-versicolor': 0 ,'Iris-virginica': 1})


In [39]:
x_bin_train,x_bin_test,y_bin_test,y_bin_test = train_test_split(
    xx,xxx,test_size=0.2,random_state=42,stratify=xxx
)

In [44]:
model = LogisticRegression()

_ =model.fit(xx,xxx)

final1 = model.predict(x_bin_test)
final2 = accuracy_score(y_bin_test,final1)

print(f"최최종:,{final2:.4f}")

최최종:,0.7000


In [12]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
df = pd.read_csv("Iris.csv")
df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [ ]:
df_binary = df[df['Species'].isin(['Iris-versicolor','Iris-virginica'])]
X_binary = df_binary[['SepalLengthCm','SepalWidthCm']]  
y_binary = df_binary['Species'].map({'Iris-versicolor': 0, 'Iris-virginica' : 1})

X_bin_train, X_bin_test, y_bin_train, y_bin_test = train_test_split(
    X_binary,y_binary, test_size=0.2, random_state=42, stratify=y_binary)

In [ ]:
binary_model = LogisticRegression()

binary_model.fit(X_bin_train,y_bin_train)

bin_test_preds = binary_model.predict(X_bin_test)

bin_accuracy = accuracy_score(y_bin_test,bin_test_preds)

print(f"TEST: {bin_accuracy:.4f}")

TEST: 0.7000


### 필수 2 : 꽃받침 크기를 활용한 다중 분류 및 교차 검증
#### 문제 2-1 :  전체 품종 다중 분류 및  K-Fold 교차 검증 수행

**요구사항**

---

1. 세 가지 품종의 데이터를 모두 사용하며, 입력 변수로 꽃받침 길이(SepalLengthCm)와 꽃받침 너비(SepalWidthCm) 열을, 정답 변수로는 품종(Species) 열을 지정하세요.
2. 데이터를 훈련 세트 80%, 테스트 세트 20% 비율로 분할하세요. 난수 고정값은 42로 설정하고 층화 추출을 적용하세요.
3. 데이터를 조각내기 전에 무작위로 섞어주는 shuffle 옵션을 True로 설정하고, 조각의 개수를 5로 지정한 케이폴드(K-Fold) 객체를 선언하세요. 난수 고정값은 42로 유지하세요.
4. 다중 분류를 위한 로지스틱 회귀 모델을 선언하고, 오직 훈련 세트 내부에서만 5-Fold 교차 검증을 실행하여 각 조각별 정확도를 구하세요.
5. 다섯 개 폴드의 정확도 수치들과 이들의 평균 검증 정확도를 각각 화면에 출력하세요.
6. 교차 검증을 마친 후 해당 다중 분류 모델을 전체 훈련 데이터로 재학습시키고, 최종 테스트 세트를 적용했을 때의 최종 테스트 정확도를 출력하세요.

**출력 및 검증 방법**
출력 결과 화면에 다섯 개 폴드의 각 정확도 배열과 평균 검증 정확도 수치, 그리고 최종 테스트 정확도가 순서대로 출력되는지 확인하세요. 실행 코드가 담긴 셀과 하단에 표시된 세 가지 성능 지표 수치들이 포함된 화면 전체를 캡처하여 작성된 코드와 함께 제출하세요.

In [23]:
X_multi = df[['SepalLengthCm', 'SepalWidthCm']]
y_multi = df['Species'].map({'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2})
X_multi.head()

,SepalLengthCm,SepalWidthCm
0,5.1,3.5
1,4.9,3.0
2,4.7,3.2
3,4.6,3.1
4,5.0,3.6


In [25]:
X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(
    X_multi,y_multi, test_size=0.2, random_state=42, stratify=y_multi)

In [51]:
multi_model = LogisticRegression()


kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(multi_model, X_multi_train, y_multi_train , cv=kf)


multi_model.fit(X_multi_train,y_multi_train)

multi_test_preds = multi_model.predict(X_multi_test)

multi_accuracy = accuracy_score(y_multi_test, multi_test_preds)
print("최종 정확도:",multi_accuracy)

최종 정확도: 0.7333333333333333


### 심화 1 : 특성 변수 확장을 통한 종합 다중 분류 성능 비교
#### 문제 3-1 : 4개 특성 전체를 활용한 모델 학습 및 성능 변화 관찰

**요구사항**

---

1. 입력 변수에 기존의 꽃받침 정보 두 개와 더불어 꽃잎 길이(PetalLengthCm)와 꽃잎 너비(PetalWidthCm)까지 총 네 개의 열을 모두 포함하세요. 정답 변수는 품종 열로 유지하세요.
2. 데이터를 훈련 세트 80%, 테스트 세트 20% 비율로 동일하게 분리하세요. 이전 문제들과 일관성을 갖도록 난수 고정값은 42, 층화 추출을 설정하세요.
3. 로지스틱 회귀 모델을 생성한 뒤 특성이 확장된 새로운 훈련 세트를 바탕으로 학습을 진행하세요.
4. 학습이 완료되면 최종 테스트 세트를 투입하여 최종 테스트 정확도를 산출하세요.
5. 앞서 필수 2번 문항에서 꽃받침 두 개만 썼을 때의 최종 테스트 정확도 수치와, 본 심화 문제에서 네 개 변수를 모두 썼을 때의 최종 테스트 정확도 수치를 나란히 출력하여 비교할 수 있도록 코드를 작성하세요.

**출력 및 검증 방법**
두 개의 변수를 사용했을 때의 성능과 네 개의 변수를 종합했을 때의 성능이 화면에 명확하게 구분되어 출력되는지 확인하세요. 특성 확장에 따른 성능 변화 결과가 드러난 최종 실행 영역 전체를 누락 없이 캡처하여 작성된 코드와 함께 제출하세요.

In [ ]:
X_multi = df[['PetalLengthCm','PetalWidthCm','SepalLengthCm','SepalWidthCm']]
y_multi = df['Species']
X_multi.head()

,PetalLengthCm,PetalWidthCm,SepalLengthCm,SepalWidthCm
0,1.4,0.2,5.1,3.5
1,1.4,0.2,4.9,3.0
2,1.3,0.2,4.7,3.2
3,1.5,0.2,4.6,3.1
4,1.4,0.2,5.0,3.6


In [63]:
x_bin_train,x_bin_test,y_bin_train,y_bin_test = train_test_split(X_multi,y_multi,test_size=0.2, random_state=42, stratify=y_multi)

model = LogisticRegression()
model.fit(x_bin_train,y_bin_train)

preds = model.predict(x_bin_test)
accur = accuracy_score(y_bin_test,preds)

print("최종 정확도:",multi_accuracy)
print("최종:",accur)

최종 정확도: 0.7333333333333333
최종: 0.9666666666666667
